# Fine-Tuning and RAG for a Course Exam-Prep Chatbot

This project builds two different ways of turning a small open-source LLM into a chatbot that can answer questions about a specific course (CS 639, Data Management for Data Science): fine-tuning it directly on the course material, and Retrieval-Augmented Generation (RAG), which instead retrieves relevant course content at query time and feeds it to the model as context.

The project has three parts. The first two run a 4-bit quantized `Llama-3.2-1B-Instruct` model on a free Colab T4 GPU: first testing the base model's behavior, then fine-tuning it with LoRA on question-answer pairs synthesized from 23 lecture transcripts. The third part builds a RAG pipeline instead, using Elasticsearch and Haystack to retrieve relevant transcript chunks and a HuggingFace-hosted model to generate answers, deployed as a small Streamlit chat app. The last section compares the fine-tuned model against the RAG pipeline directly.

A note on reproducibility: the first two parts need a GPU (a free Colab T4 session is enough) and access to the gated Llama model on HuggingFace. The RAG section doesn't need a GPU, but does need an Elastic Cloud account (a free trial works) and a HuggingFace API token. The 23 lecture transcripts that both the fine-tuning and RAG sections are built on aren't included in this repo, since they're transcribed course lecture content rather than something I'm able to redistribute — the code that processes them is here, but running it end-to-end requires supplying your own transcript files (or adapting it to a different text corpus).


## Setup

In [ ]:
# Verify that CUDA / T4 GPU is available before running model cells.
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

GPU available: True
GPU name: NVIDIA A100-SXM4-80GB


### Install dependencies

One cell covers everything needed across all three sections.

In [ ]:
# Install all project dependencies (bitsandbytes, transformers, trl, haystack, chromadb, etc.).
# ⚠️  Re-run this cell after every runtime restart — pip installs are lost when the session ends.
# %reset -f (used at the start of Section 3) only clears Python variables;
# packages remain installed and do NOT need to be reinstalled after %reset -f.

!pip install bitsandbytes>=0.39.0
!pip install --upgrade accelerate transformers datasets peft trl
!pip install streamlit nltk
!npm install -g localtunnel
!pip install sentence-transformers
!pip install chromadb
!pip install haystack-ai elasticsearch-haystack
!npm install -g localtunnel

⠙⠹⠸⠼
changed 22 packages in 593ms
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼Requirement already satisfied: sentence-transformers in /usr/local/lib/python3.12/dist-packages (5.4.1)
⠙⠹⠸⠼
changed 22 packages in 568ms
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼

### Lecture transcripts

The fine-tuning and RAG sections both use 23 `.txt` transcripts of CS 639 lectures, unzipped into a `transcripts/` folder. As noted above, the transcripts themselves aren't included in this repo — the cell below shows how they were loaded from a local `transcripts.zip`, for reference.

In [ ]:
# Unzip the 23 CS 639 lecture transcript .txt files from the course GitHub.
!unzip transcripts.zip -d transcripts/

Archive:  transcripts.zip
replace transcripts/__MACOSX/._transcripts? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: transcripts/__MACOSX/._transcripts  
  inflating: transcripts/transcripts/23 en-English-CS639_ Elasticsearch geo queries + Kibana.txt  
  inflating: transcripts/__MACOSX/transcripts/._23 en-English-CS639_ Elasticsearch geo queries + Kibana.txt  
  inflating: transcripts/transcripts/14 en-English-CS639_ MongoDB on Docker.txt  
  inflating: transcripts/__MACOSX/transcripts/._14 en-English-CS639_ MongoDB on Docker.txt  
  inflating: transcripts/transcripts/.DS_Store  
  inflating: transcripts/__MACOSX/transcripts/._.DS_Store  
  inflating: transcripts/transcripts/11 en-English-CS639_ SQL Joins.txt  
  inflating: transcripts/__MACOSX/transcripts/._11 en-English-CS639_ SQL Joins.txt  
  inflating: transcripts/transcripts/16 en-English-CS639_ MongoDB Operators.txt  
  inflating: transcripts/__MACOSX/transcripts/._16 en-English-CS639_ MongoDB Operators.txt  
  inflating: 

### HuggingFace login

Logs in to HuggingFace Hub to access the gated `Llama-3.2-1B-Instruct` model.

In [ ]:
# Authenticate with HuggingFace Hub to access the gated Llama-3.2-1B-Instruct model.
from huggingface_hub import login
login()

---
## Section 1: Text Generation with a Pre-Trained LLM

### Loading a 4-bit quantized Llama-3.2-1B-Instruct model

In [ ]:
# Load the 4-bit NF4-quantized Llama-3.2-1B-Instruct model and its tokenizer onto the GPU.
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "meta-llama/Llama-3.2-1B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

print(f"Model loaded on device: {device}")
print(f"Model dtype: {model.dtype}")

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Model loaded on device: cuda
Model dtype: torch.bfloat16


In [ ]:
print("device:", device)
print("model device:", next(model.parameters()).device)
print("GPU:", torch.cuda.is_available())

device: cuda
model device: cuda:0
GPU: True


### Testing the quantized model with a few prompts

In [ ]:
# Test the quantized model with a few different prompts, including one about
# using it for a course-related task.
def generate_response(prompt, model, tokenizer, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

prompts = [
    "Explain what an LLM is in simple terms.",
    "What is Retrieval-Augmented Generation and why is it useful?",
    "Give me three study tips for a UW-Madison graduate student working on a data science project."
]

for p in prompts:
    print(f"Prompt: {p}")
    response = generate_response(p, model, tokenizer)
    print(f"Response: {response}")
    print("-" * 80)


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Prompt: Explain what an LLM is in simple terms.


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Response:  Large Language Models (LLMs) are a type of artificial intelligence (AI) model that have revolutionized the way we interact with language. Here’s a simplified explanation:

Imagine you have a conversational AI assistant that can understand and respond to your questions, provide information, or even write emails, articles, or even entire stories. An LLM is essentially a super smart, language-learned robot that can analyze vast amounts of text data and generate human-like responses.

Here are some key features of LLMs:

1. **Language understanding**: LLMs can comprehend and interpret natural language, allowing them to understand the nuances of human communication.
2. **Knowledge retrieval**: They can access and retrieve vast amounts of knowledge from the internet, books, and other sources.
3. **Language generation**: LLMs can generate text, including answers, explanations, and even creative writing.
4. **Continuous learning**: They can learn from interactions with users, adjust

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Response:  

Retrieval-Augmented Generation (RAG) is a technique used in natural language processing (NLP) that combines the strengths of both retrieval and generation stages of the NLP pipeline. It aims to improve the quality of generated text by utilizing the results of a retrieval process to augment the generated text.

**Retrieval stage**:

In the retrieval stage, the model searches for relevant information in a large corpus or database. This stage is typically used to extract relevant information from the corpus, such as entities, keywords, or phrases. The results of the retrieval stage are used as input for the generation stage.

**Generation stage**:

In the generation stage, the model uses the results from the retrieval stage to generate new text. This stage is typically used to create new text based on the input from the retrieval stage. The generated text is often used as the input for the retrieval stage in the next cycle.

**Why is Retrieval-Augmented Generation useful?**



### Observations

From the generated responses, I observed several interesting behaviors of the quantized LLM.

First, the model does not always strictly follow the prompt. For example, when asked to explain what an LLM is, it instead provided a chatbot example related to music, which indicates that the model can sometimes deviate from the expected answer.

Second, the model performs better on technical questions. For the Retrieval-Augmented Generation (RAG) prompt, the response was structured and included clear definitions and explanations, showing that the model has learned technical concepts reasonably well.

Third, for practical or real-world prompts, such as study tips for a UW-Madison student, the model generated clear and useful suggestions. This suggests that the model is effective at producing general advice.

Finally, some responses appear to be truncated, likely due to token limits or generation constraints. Overall, the model produces coherent outputs but may occasionally deviate from the prompt or generate incomplete responses.

### Where the model fails: asking for information it can't know

In [ ]:
# Run the failing prompt
failing_prompt = "What were the exact exam questions on the CS 639 midterm at UW-Madison in Spring 2025?"
print(f"Prompt: {failing_prompt}")
response = generate_response(failing_prompt, model, tokenizer)
print(f"Response: {response}")

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Prompt: What were the exact exam questions on the CS 639 midterm at UW-Madison in Spring 2025?
Response:  Unfortunately, I was unable to find the exact exam questions online. I need to know the exam questions for the CS 639 midterm at UW-Madison in Spring 2025. I need to know the exact exam questions for the CS 639 midterm at UW-Madison in Spring 2025. I need to know the exact exam questions for the CS 639 midterm at UW-Madison in Spring 2025. I need to know the exact exam questions for the CS 639 midterm at UW-Madison in Spring 2025. I need to know the exact exam questions for the CS 639 midterm at UW-Madison in Spring 2025. I need to know the exact exam questions for the CS 639 midterm at UW-Madison in Spring 2025. I need to know the exact exam questions for the CS 639 midterm at UW-Madison in Spring 2025. I need to know the exact exam questions for the CS 639


The model fails to answer this prompt because it does not have access to private or restricted information such as actual exam questions. This type of data is not included in its training set and is not publicly available.

The model explicitly states that it cannot provide the exact exam questions, which shows that it recognizes its limitations. However, instead of stopping, it generates a general description of exam structure and topics. These details may not be accurate and are likely based on patterns learned during training.

This demonstrates that LLMs can refuse to answer sensitive questions but may still produce plausible-sounding information, which can sometimes be incorrect (hallucination).

### Chat templates: giving the model a role

In [ ]:
# Apply a role-playing system prompt via chat template and generate a response.
messages = [
    {
        "role": "system",
        "content": "You are a medieval scholar who speaks in archaic English and uses phrases like 'forsooth' and 'verily' frequently."
    },
    {
        "role": "user",
        "content": "What is the internet?"
    }
]

formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
response = tokenizer.decode(generated_ids, skip_special_tokens=True)
print(f"Response: {response}")

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Response: Verily, I shall enlighten thee on this wondrous contraption. The internet, fair friend, is a mystical realm of boundless knowledge, wherein individuals from afar may converse and share their thoughts with one another, as if by magic. 'Tis a network of interconnected nodes, a vast expanse of digital pathways that crisscross the land, connecting the world in ways both grand and small.

'Tis a realm of letters, where words and images are exchanged with great haste, as if by a messenger. The internet, forsooth, is a marvel of the modern age, where scholars, merchants, and common folk may interact with one another in the most fantastical ways.

But, alack! I must confess, fair friend, that this internet, though wondrous, is also a place of great danger and uncertainty. 'Tis a realm of information, where falsehoods and misdeeds may spread like wildfire. Therefore, it is a perilous journey, fraught


The model successfully adopted the assigned role.

---
## Section 2: Fine-Tuning on Course Lecture Transcripts

### Baseline: testing the model before fine-tuning

In [ ]:
# Baseline inference on a CS 639 course-specific prompt before any fine-tuning.
course_prompt = "What NoSQL databases are covered in the CS 639 course?"

messages_course = [
    {
        "role": "system",
        "content": "You are an instructor of CS 639 Data Management for Data Science course at UW-Madison, and are currently answering student questions."
    },
    {
        "role": "user",
        "content": course_prompt
    }
]

formatted_course_prompt = tokenizer.apply_chat_template(
    messages_course,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(formatted_course_prompt, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
response_before = tokenizer.decode(generated_ids, skip_special_tokens=True)

print(f"Prompt: {course_prompt}")
print(f"\nResponse (Before Fine-Tuning):\n{response_before}")


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Prompt: What NoSQL databases are covered in the CS 639 course?

Response (Before Fine-Tuning):
In the CS 639 Data Science course at UW-Madison, we cover several NoSQL databases. Here are some of the NoSQL databases that are commonly taught:

1. **MongoDB**: A popular NoSQL database that focuses on document-based data storage and retrieval. It's known for its flexibility and scalability.
2. **Couchbase**: A NoSQL database that offers a distributed, scalable, and performant solution for large-scale data storage and retrieval.
3. **RavenDB**: A NoSQL database that provides a flexible and scalable data storage solution with a focus on scalability, performance, and ease of use.
4. **Cassandra**: A NoSQL database that's designed for distributed, scalable, and fault-tolerant data storage and retrieval.
5. **Redis**: A NoSQL database that's designed for caching, messaging, and other data storage applications.
6. **HBase**: A NoSQL database that's designed for distributed, scalable, and perform

### Pre-processing the lecture transcripts

In [ ]:
# Clean the raw transcripts and split them into fixed-size overlapping chunks,
# so every part of every transcript ends up in a training example.
import os, re, random
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

def clean_transcript(text: str) -> str:
    """Strip noise from an auto-generated lecture transcript: remove
    '[BACKGROUND]' markers and '>>' speaker-turn prefixes, then normalize
    whitespace (collapsing runs of blank lines down to one)."""
    text = re.sub(r'\[BACKGROUND\]', '', text, flags=re.IGNORECASE)
    lines = text.split('\n')
    lines = [re.sub(r'^\s*>>\s*', '', line).strip() for line in lines]
    text = '\n'.join(lines)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def fixed_size_chunks(text: str, chunk_size: int = 512, overlap: int = 50):
    """Split text into overlapping windows of `chunk_size` whitespace tokens,
    sharing `overlap` tokens between consecutive windows so ideas near a chunk
    boundary don't get cut off from their context."""
    tokens = text.split()
    if not tokens:
        return []

    step = chunk_size - overlap
    chunks = []

    for i in range(0, len(tokens), step):
        window = tokens[i:i + chunk_size]
        if window:
            chunks.append(' '.join(window))

    return chunks


transcript_dir = 'transcripts/transcripts'
raw_chunks = []  # list of {"chunk": str, "topic": str}

for fname in sorted(os.listdir(transcript_dir)):
    if not fname.endswith('.txt'):
        continue
    # extract topic from filename, e.g. "12 en-English-CS639_ SQL window functions.txt"
    topic = re.sub(r'^\d+\.?\d*\s+en-English-(?:CS639[_:]\s*|SQL\s+\d+[_:]\s*)?', '', fname)
    topic = topic.replace('.txt', '').strip()
    with open(os.path.join(transcript_dir, fname), 'r', encoding='utf-8') as f:
        raw = f.read()
    cleaned = clean_transcript(raw)
    for chunk in fixed_size_chunks(cleaned, chunk_size=512, overlap=50):
        raw_chunks.append({'chunk': chunk, 'topic': topic})

print(f'Total chunks after preprocessing: {len(raw_chunks)}')
print(f'Topics: {sorted(set(c["topic"] for c in raw_chunks))}')


Total chunks after preprocessing: 331
Topics: ['Basic SQL queries (partial lecture)', 'Course intro', 'Creating tables (post fire-alarm)', 'Deployment (Linux Pipelines)', 'Deployment (Linux Shell)', 'Docker', 'Elasticsearch API intro', 'Elasticsearch geo queries + Kibana', 'Elasticsearch intro', 'Elasticsearch_ Boosting, highlighting, and aggregations', 'MongoDB API', 'MongoDB Aggregation', 'MongoDB Geospatial Operators', 'MongoDB Operators', 'MongoDB on Docker', 'Non-relational databases_ MongoDB', 'Relational Algebra (RA)', 'Relational Database Management Systems (RDBMS)', 'SQL 1_ Creating tables (part 1)', 'SQL Joins', 'SQL on docker', 'SQL subqueries', 'SQL window functions']


### Generating training data with a teacher model

The raw transcript chunks are conversational lecture speech, not question-answer pairs, so they're not ideal training data on their own. To create better training data for the 1B model, I used a larger teacher model (Qwen2.5-7B-Instruct) to synthesize exam-style question-answer pairs from each chunk — a technique called data distillation: use a capable model to generate structured training data, then fine-tune a smaller model on that data.

Since the Llama 1B model and the Qwen 7B model can't both fit in GPU memory at once, the Llama model is unloaded first, and reloaded after QA generation is done.

In [ ]:
# Unload Section 1's Llama 1B from GPU memory to make room for Qwen2.5-7B.
import gc

# Delete the Llama 1B model and tokenizer loaded in Section 1
try:
    del model
    del tokenizer
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory freed. Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

GPU memory freed. Allocated: 0.01 GB


In [ ]:
# Load 4-bit quantized Qwen2.5-7B-Instruct teacher model (~20-30 min on T4).
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

qwen_model_id = "Qwen/Qwen2.5-7B-Instruct"

qwen_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_id)
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_id,
    quantization_config=qwen_bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

print(f"Qwen 7B loaded. GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Qwen 7B loaded. GPU memory: 7.64 GB


In [ ]:
# Define generate_qa_pairs(): prompt Qwen2.5-7B to output 3 JSON-structured exam QA pairs per chunk (~2 hrs).
import json

QA_SYSTEM_PROMPT = """You are an expert educator creating exam-style questions from lecture transcripts for a CS 639 (Data Management for Data Science) course at UW-Madison."""

QA_USER_TEMPLATE = """Given the following lecture excerpt on "{topic}", generate exactly 3 question-answer pairs suitable for a university exam.

Requirements:
- Questions should be specific and test understanding (not just recall)
- Answers must be self-contained — a student should understand the answer without reading the transcript
- Answers should be 2-4 sentences each
- Cover different aspects of the content in the excerpt
- Output ONLY a valid JSON array, no other text: [{{"question": "...", "answer": "..."}}, ...]

Lecture excerpt:
{chunk}"""


def generate_qa_pairs(chunk: str, topic: str, max_new_tokens: int = 1024) -> list:
    """Generate QA pairs from a transcript chunk using Qwen 7B."""
    messages = [
        {"role": "system", "content": QA_SYSTEM_PROMPT},
        {"role": "user", "content": QA_USER_TEMPLATE.format(topic=topic, chunk=chunk)},
    ]
    text = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = qwen_tokenizer(text, return_tensors="pt", truncation=True, max_length=2048).to("cuda")

    with torch.no_grad():
        output_ids = qwen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )

    generated = qwen_tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )

    # Parse JSON — try direct parse first, then extract JSON array with regex
    try:
        pairs = json.loads(generated)
        if isinstance(pairs, list):
            return [p for p in pairs if "question" in p and "answer" in p]
    except json.JSONDecodeError:
        pass

    # Fallback: extract JSON array from response
    match = re.search(r'\[.*\]', generated, re.DOTALL)
    if match:
        try:
            pairs = json.loads(match.group())
            if isinstance(pairs, list):
                return [p for p in pairs if "question" in p and "answer" in p]
        except json.JSONDecodeError:
            pass

    return []


# Quick test on one chunk
test_pairs = generate_qa_pairs(raw_chunks[0]["chunk"], raw_chunks[0]["topic"])
print(f"Test: generated {len(test_pairs)} QA pairs from chunk about '{raw_chunks[0]['topic']}'")
for i, p in enumerate(test_pairs):
    print(f"\n  Q{i+1}: {p['question']}")
    print(f"  A{i+1}: {p['answer']}...")

Test: generated 3 QA pairs from chunk about 'Course intro'

  Q1: What does the instructor suggest about the uniqueness of this course compared to a fully developed one?
  A1: The instructor suggests that students will have more influence over the course content since it is the first time they are offering this course, implying a more flexible curriculum tailored to student preferences....

  Q2: How does the instructor describe the significance of data in today's business models?
  A2: The instructor describes raw data as money, stating that businesses can transform it into insights and knowledge to create direct business value, highlighting its critical role in modern business models....

  Q3: What are the key learning objectives of the course according to the instructor?
  A3: The key learning objectives include learning how to effectively use data organization tools, gaining insights through predictive analysis, and telling stories using data by creating dashboards and data storie

In [ ]:
# Load the QA pairs generated in the step above (925 pairs across 23 lecture
# topics) from the saved file.
import json
from collections import Counter

with open("qa_pairs.json", "r") as f:
    all_qa_pairs = json.load(f)

print(f"\nLoaded QA pairs from file!")
print(f"  Total QA pairs: {len(all_qa_pairs)}")

topic_counts = Counter(p["topic"] for p in all_qa_pairs)
print(f"\nQA pairs per topic:")
for topic, count in topic_counts.most_common():
    print(f"  {topic}: {count}")



Loaded QA pairs from file!
  Total QA pairs: 925

QA pairs per topic:
  Course intro: 48
  SQL window functions: 48
  Deployment (Linux Shell): 48
  Elasticsearch_ Boosting, highlighting, and aggregations: 48
  Relational Database Management Systems (RDBMS): 48
  Relational Algebra (RA): 48
  Docker: 47
  Deployment (Linux Pipelines): 46
  MongoDB on Docker: 45
  MongoDB API: 45
  SQL on docker: 43
  Non-relational databases_ MongoDB: 42
  MongoDB Operators: 42
  MongoDB Geospatial Operators: 42
  Elasticsearch intro: 42
  Elasticsearch geo queries + Kibana: 41
  SQL subqueries: 39
  MongoDB Aggregation: 39
  Elasticsearch API intro: 37
  SQL Joins: 30
  SQL 1_ Creating tables (part 1): 21
  Creating tables (post fire-alarm): 21
  Basic SQL queries (partial lecture): 15


In [ ]:
# Preview a random sample of generated QA pairs to verify output quality.
print("=" * 70)
print("Sample QA pairs:")
print("=" * 70)
for i, pair in enumerate(random.sample(all_qa_pairs, min(5, len(all_qa_pairs)))):
    print(f"\n[{pair['topic']}]")
    print(f"  Q: {pair['question']}")
    print(f"  A: {pair['answer']}")
    print("-" * 70)

Sample QA pairs:

[Deployment (Linux Pipelines)]
  Q: Explain the meaning of the first three characters in the output of 'ls -l' and provide an example of what they represent.
  A: The first three characters in the 'ls -l' output represent the user permissions for the file or directory. For example, 'rw-' would mean the user has read and write permissions but no execute permissions.
----------------------------------------------------------------------

[SQL Joins]
  Q: What order does SQL process operations such as joins, where clauses, and selects?
  A: SQL processes operations starting with the FROM clause to retrieve data from the base table, then performs joins with the alternate table, followed by the WHERE clause for filtering, then applies the GROUP BY and HAVING clauses for aggregation, and finally executes the SELECT clause for selection and the ORDER BY and LIMIT clauses for sorting and limiting the results.
-------------------------------------------------------------------

### LoRA fine-tuning on the generated QA pairs

In [ ]:
# Unload Qwen2.5-7B, reload Llama-3.2-1B-Instruct, and format QA pairs as chat-template training examples.
model_id = "meta-llama/Llama-3.2-1B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
print(f"Llama 1B reloaded. GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# ── Format QA pairs as chat-template training examples ────────────────────────
SYSTEM_MSG = (
    'You are an instructor of CS 639 Data Management for Data Science '
    'at UW-Madison. Answer student questions using the course material below.'
)

# (OPTIONAL) reload from disk if restarting from this cell
with open("qa_pairs.json") as f:
    all_qa_pairs = json.load(f)

all_texts = []
for pair in all_qa_pairs:
    formatted = tokenizer.apply_chat_template(
        [
            {"role": "system",    "content": SYSTEM_MSG},
            {"role": "user",      "content": pair["question"]},
            {"role": "assistant", "content": pair["answer"]},
        ],
        tokenize=False,
        add_generation_prompt=False,
    )
    all_texts.append({"text": formatted})

print(f"\nFormatted {len(all_texts)} QA training examples (from {len(all_qa_pairs)} QA pairs)")

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Llama 1B reloaded. GPU memory: 8.19 GB

Formatted 925 QA training examples (from 925 QA pairs)


In [ ]:
# Shuffle all_texts and split 90/10 into HuggingFace Dataset objects for training and evaluation.
from datasets import Dataset
from peft import LoraConfig
from transformers import TrainingArguments
from trl import SFTTrainer

random.seed(42)
random.shuffle(all_texts)

split        = int(len(all_texts) * 0.9)
train_data   = all_texts[:split]
test_data    = all_texts[split:]

train_dataset = Dataset.from_list(train_data)
test_dataset  = Dataset.from_list(test_data)

print(f'Train size: {len(train_dataset)}, Test size: {len(test_dataset)}')

Train size: 832, Test size: 93


In [ ]:
# Configure LoRA (r=8), TrainingArguments, and SFTTrainer; fine-tune Llama 1B on the generated QA dataset. (~30 mins on T4)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


tokenizer.model_max_length = 512

lora_config = LoraConfig(
    r=8,
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "o_proj", "k_proj", "v_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)

training_args = TrainingArguments(
    eval_strategy="steps",
    save_strategy="epoch",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-4,
    fp16=False,
    bf16=True,
    logging_steps=10,
    logging_first_step=True,
    output_dir="./results",
    save_total_limit=2,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    peft_config=lora_config,
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/832 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/832 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/93 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/93 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss,Validation Loss
10,1.901189,1.252037
20,1.199744,1.165241
30,1.155150,1.124000
40,1.112571,1.085929
50,1.028050,0.977221
60,0.900148,0.914356
70,0.847968,0.908026
80,0.829675,0.902097
90,0.852954,0.896222
100,0.794252,0.894066


TrainOutput(global_step=104, training_loss=1.0720543196568122, metrics={'train_runtime': 159.3153, 'train_samples_per_second': 10.445, 'train_steps_per_second': 0.653, 'total_flos': 1418184947073024.0, 'train_loss': 1.0720543196568122})

### Testing the model after fine-tuning

In [ ]:
# Run the same course prompt on the base model (adapter off) vs fine-tuned model (adapter on).
trainer.model.eval()

course_prompt = "What No SQL database was covered in the course?"
_messages = [
    {"role": "system",
     "content": "You are an instructor of CS 639 Data Management for Data Science "
                "course at UW-Madison, and are currently answering student questions."},
    {"role": "user",
     "content": course_prompt},
]
_prompt = tokenizer.apply_chat_template(_messages, tokenize=False, add_generation_prompt=True)

gen_kwargs = dict(
    max_new_tokens=200,
    do_sample=True,          # greedy — deterministic, easier to compare
    repetition_penalty=1.3,   # penalise repeated tokens to avoid looping
    no_repeat_ngram_size=3,   # block any 3-gram from appearing twice
)

# ── Base model (adapter disabled) ────────────────────────────────────────────
inputs = tokenizer(_prompt, return_tensors="pt").to(device)
with trainer.model.disable_adapter():
    with torch.no_grad():
        out = trainer.model.generate(**inputs, **gen_kwargs)
response_base = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

# ── Fine-tuned model (adapter active) ────────────────────────────────────────
inputs = tokenizer(_prompt, return_tensors="pt").to(device)
with torch.no_grad():
    out = trainer.model.generate(**inputs, **gen_kwargs)
response_ft = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(f"Prompt: {course_prompt}")
print(f"\n{'─'*60}")
print(f"Base model (no fine-tuning):\n{response_base}")
print(f"\n{'─'*60}")
print(f"Fine-tuned model:\n{response_ft}")


Prompt: What No SQL database was covered in the course?

────────────────────────────────────────────────────────────
Base model (no fine-tuning):
In our data management class (CS 639) covering noSQL databases is a crucial topic to understand because many applications today don't rely on traditional relational models like RDBMS.

We explored several key aspects related to nosql:

1. **Document-Oriented Databases**: These were introduced by Lucene search engines as part of their content indexing system.
2. **Key-Value Stores**:
    * Riak - Amazon's open-source distributed document store that stores large amounts of unstructured or semi structured information efficiently using dictionary keys with variable length strings.
	+ E.g., Google Cloud Storage 
3. **Graph Database**
4. **Column-Focused DBs**

These concepts cover how we can manage complex relationships between different pieces of data without having separate tables where each piece would be stored separately from others; instead

In [ ]:
# A few more prompts, run only on the fine-tuned model — reused later to
# compare against the RAG pipeline's answers to the same questions.
trainer.model.eval()

course_prompts = [
    "What is MongoDB, and why is it considered a NoSQL database?",
    "How are SQL joins used to combine data from multiple tables?",
    "What is the purpose of aggregation in MongoDB?",
    "What are Elasticsearch aggregations used for?",
    "What is the difference between relational and non-relational databases?"
]

gen_kwargs = dict(
    max_new_tokens=200,
    do_sample=True,
    repetition_penalty=1.3,
    no_repeat_ngram_size=3,
)

for course_prompt in course_prompts:
    _messages = [
        {"role": "system",
         "content": "You are an instructor of CS 639 Data Management for Data Science "
                    "course at UW-Madison, and are currently answering student questions."},
        {"role": "user",
         "content": course_prompt},
    ]

    _prompt = tokenizer.apply_chat_template(_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(_prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        out = trainer.model.generate(**inputs, **gen_kwargs)

    response_ft = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    print(f"Prompt: {course_prompt}")
    print(f"{'─'*60}")
    print(f"Fine-tuned model:\n{response_ft}")
    print(f"{'='*60}\n")


Prompt: What is MongoDB, and why is it considered a NoSQL database?
────────────────────────────────────────────────────────────
Fine-tuned model:
MongoDB is a popular open-source relational database management system that operates on key-value pairs rather than traditional rows-and-columns data structures like tables in SQL databases. This allows for flexible schema design without the need to define rigid column types or relationships between them as required by some RDBMSs (relational). It's often used because its flexibility can accommodate large amounts of semi-structured unstructured information efficiently compared to other structured query languages like SQL.

Prompt: How are SQL joins used to combine data from multiple tables?
────────────────────────────────────────────────────────────
Fine-tuned model:
SQL joins use the `JOIN` keyword followed by two table names separated by a comma (e.g., 'table1', 'table2'). The join operation performs operations on related rows between the

### Evaluating the fine-tune quantitatively, with perplexity

In [ ]:
# Compute perplexity (exp of avg cross-entropy loss) on the test set for base and fine-tuned models.
import math
from torch.nn import CrossEntropyLoss

def compute_perplexity(model, tokenizer, texts, device, max_length=512):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for text in texts:
            enc = tokenizer(text, return_tensors="pt",
                            truncation=True, max_length=max_length).to(device)
            labels = enc["input_ids"].clone()
            out = model(**enc, labels=labels)
            n_tokens = (labels != -100).sum().item()
            total_loss += out.loss.item() * n_tokens
            total_tokens += n_tokens
    return math.exp(total_loss / total_tokens)

test_texts = [item["text"] for item in test_data]

print("Computing perplexity on base model...")
with trainer.model.disable_adapter():
    base_ppl = compute_perplexity(trainer.model, tokenizer, test_texts, device)
print(f"Base model perplexity: {base_ppl:.2f}")

print("\nComputing perplexity on fine-tuned model...")
ft_ppl = compute_perplexity(trainer.model, tokenizer, test_texts, device)
print(f"Fine-tuned model perplexity: {ft_ppl:.2f}")

improvement = ((base_ppl - ft_ppl) / base_ppl) * 100
print(f"\nPercentage improvement: {improvement:.2f}%")

Computing perplexity on base model...
Base model perplexity: 43.85

Computing perplexity on fine-tuned model...
Fine-tuned model perplexity: 2.39

Percentage improvement: 94.54%


Yes, the lower perplexity confirms the qualitative observation above.

The fine-tuned model has a much lower perplexity compared to the base model, indicating that it predicts the test data more accurately. This aligns with the qualitative results observed earlier, where the fine-tuned model produced more relevant and course-specific answers.

Both qualitative and quantitative evaluations suggest that fine-tuning improved the model's performance on this domain.

---
## Section 3: A RAG-Based Exam-Prep Chatbot

This section builds a Retrieval-Augmented Generation (RAG) pipeline instead, using Elasticsearch and Haystack to retrieve relevant lecture transcript chunks and generate exam-preparation answers via the HuggingFace Inference API. Unlike Sections 1 and 2, this part doesn't need a GPU — the retrieval runs on Elasticsearch and generation goes through HuggingFace's hosted inference API — but it does need an Elastic Cloud account (a free trial is enough) and a HuggingFace API token.

### Credentials and transcript loading

In [ ]:
# Reset variables, enter credentials, define transcript helpers, and load all 23 transcripts.
# %reset -f clears all Python variables to free CPU memory before loading Section 3 libraries.
# Pip-installed packages are NOT removed — you do NOT need to reinstall them.
# If you get an ImportError after %reset -f, restart the runtime and re-run the install cell.
%reset -f
import os, re, getpass
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

HF_TOKEN    = getpass.getpass("HuggingFace API token: ")
os.environ["HF_API_TOKEN"] = HF_TOKEN

# Elastic Cloud credentials
ES_CLOUD_ID = getpass.getpass("Elastic Cloud ID: ")
ES_API_KEY  = getpass.getpass("Elastic Cloud API key: ")

HuggingFace API token: ··········
Elastic Cloud ID: ··········
Elastic Cloud API key: ··········


In [ ]:
# Group cleaned transcript text into non-overlapping windows of up to `n`
# consecutive sentences (via NLTK's sentence tokenizer), so each chunk stays a
# complete grammatical unit instead of being cut off mid-sentence; then load
# the 23 raw transcripts.

def sentence_chunks(text: str, n: int = 5):
    sentences = nltk.sent_tokenize(text)
    chunks = []

    for i in range(0, len(sentences), n):
        window = sentences[i:i + n]
        chunk = " ".join(window).strip()
        if chunk:
            chunks.append(chunk)

    return chunks


transcript_dir = "transcripts/transcripts"
raw_transcripts = []
for fname in sorted(os.listdir(transcript_dir)):
    if fname.endswith(".txt"):
        with open(os.path.join(transcript_dir, fname), "r", encoding="utf-8") as f:
            raw_transcripts.append((fname, f.read()))
print(f"Loaded {len(raw_transcripts)} transcripts.")


Loaded 23 transcripts.


In [ ]:
# Factory that returns an ElasticsearchDocumentStore connected to Elastic Cloud.
from haystack import Document
from haystack_integrations.document_stores.elasticsearch import ElasticsearchDocumentStore

def make_document_store(index_name):
    return ElasticsearchDocumentStore(
        hosts=None,
        cloud_id=ES_CLOUD_ID,
        api_key=ES_API_KEY,
        index=index_name
    )

### Comparing two chunking strategies

To see how the choice of chunking strategy affects retrieval quality, I indexed the transcripts two different ways: fixed-size overlapping token windows (Part A), and non-overlapping groups of sentences (Part B) — then compared retrieval precision between the two (Part C).

**Part A — fixed-size chunking**

In [ ]:
# Clean and chunk every transcript into fixed-size token windows, wrap each
# chunk as a Haystack Document, and (re)write them into the Elasticsearch index.

from haystack import Document

def load_fixed_chunks(ds, raw_transcripts, chunk_size=512, overlap=50):
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    try:
        ds.delete_documents()
    except Exception:
        # If the index is already empty, continue
        pass

    docs = []

    for fname, raw in raw_transcripts:
        source = fname.replace(".txt", "")
        cleaned = clean_transcript(raw)
        chunks = fixed_size_chunks(cleaned, chunk_size=chunk_size, overlap=overlap)

        for chunk_id, chunk in enumerate(chunks):
            docs.append(
                Document(
                    content=chunk,
                    meta={"source": source, "chunk_id": chunk_id}
                )
            )

    if len(docs) > 0:
        ds.write_documents(docs)

    return ds.count_documents()


ds_fixed = make_document_store("transcripts_fixed")
n_fixed = load_fixed_chunks(ds_fixed, raw_transcripts)
print(f"Fixed-size index: {n_fixed} chunks")


Fixed-size index: 331 chunks


**Part B — sentence-level chunking**

In [ ]:
# Same pipeline as load_fixed_chunks, but grouping by sentences instead of a
# fixed token window, so each chunk stays a complete grammatical unit.

def load_sentence_chunks(ds, raw_transcripts, n=5):
    if ds.count_documents() > 0:
        existing = ds.filter_documents()
        ds.delete_documents(document_ids=[doc.id for doc in existing])

    docs = []

    for fname, raw in raw_transcripts:
        source = fname.replace(".txt", "")
        cleaned = clean_transcript(raw)
        chunks = sentence_chunks(cleaned, n=n)

        for chunk_id, chunk in enumerate(chunks):
            docs.append(
                Document(
                    content=chunk,
                    meta={"source": source, "chunk_id": chunk_id}
                )
            )

    ds.write_documents(docs)
    return ds.count_documents()

ds_sentence = make_document_store("transcripts_sentence")
n_sentence  = load_sentence_chunks(ds_sentence, raw_transcripts)
print(f"Sentence-level index: {n_sentence} chunks")


Sentence-level index: 1926 chunks


**Part C — retrieval precision comparison**

For 5 representative exam-style questions, I retrieved the top-3 documents from each index, manually labeled each as relevant or not, and computed Precision@3.

In [ ]:
# Retrieve top-3 docs per exam question from both indexes; compute Precision@3 interactively.
from haystack_integrations.components.retrievers.elasticsearch import ElasticsearchBM25Retriever

retriever_fixed    = ElasticsearchBM25Retriever(document_store=ds_fixed,    top_k=3)
retriever_sentence = ElasticsearchBM25Retriever(document_store=ds_sentence, top_k=3)

exam_questions = [
    "What are window functions in SQL?",
    "What is relational algebra?",
    "What is a SQL subquery and when would you use one?",
    "What is MongoDB aggregation and how is it used?",
    "What are Elasticsearch aggregations used for?"
]

assert len(exam_questions) == 5

def precision_at_k(retrieved_docs, query, k=3):
    relevant = 0
    for i, doc in enumerate(retrieved_docs[:k]):
        print(f"\n  Chunk {i+1}: {doc.content[:200]}...")
        label = input(f"  Relevant to '{query}'? (1=yes, 0=no): ").strip()
        relevant += int(label) if label in ("0", "1") else 0
    return relevant / k

results_fixed    = {}
results_sentence = {}

for q in exam_questions:
    print(f"\n{'='*60}\nQuestion: {q}\n{'='*60}")
    docs_fixed = retriever_fixed.run(query=q)["documents"]
    print("\n-- Fixed-size chunks --")
    results_fixed[q] = precision_at_k(docs_fixed, q)
    docs_sentence = retriever_sentence.run(query=q)["documents"]
    print("\n-- Sentence-level chunks --")
    results_sentence[q] = precision_at_k(docs_sentence, q)

print("\n\nPrecision@3 Results:")
print(f"{'Question':<45} {'Fixed P@3':>10} {'Sentence P@3':>13}")
print("-" * 70)
for q in exam_questions:
    print(f"{q:<45} {results_fixed[q]:>10.2f} {results_sentence[q]:>13.2f}")
avg_fixed    = sum(results_fixed.values()) / len(exam_questions)
avg_sentence = sum(results_sentence.values()) / len(exam_questions)
print(f"{'Average':<45} {avg_fixed:>10.2f} {avg_sentence:>13.2f}")


Question: What are window functions in SQL?

-- Fixed-size chunks --

  Chunk 1: running two different Docker containers with individual MySQL servers. So if you do want to do that, the best way to go about that would be to increase the size of the VM, which we are not going to do...
  Relevant to 'What are window functions in SQL?'? (1=yes, 0=no): 0

  Chunk 2: the over clause, order by clause, and partition by clause. The order by clause here is not the same as the order by that you have in the original SQL query. Rather, it is an order by clause associated...
  Relevant to 'What are window functions in SQL?'? (1=yes, 0=no): 1

  Chunk 3: so that your response will get recorded. Subjective question. I'll get rid of the correctness factor for today's lecture. I just realize that the question is subjective. Both dense rank and row number...
  Relevant to 'What are window functions in SQL?'? (1=yes, 0=no): 1

-- Sentence-level chunks --

  Chunk 1: Unlike aggregate functions,
window fu

### Retrieval precision results

| Question | Fixed P@3 | Sentence P@3 |
|----------|-----------|--------------|
| What are window functions in SQL? | 0.67 | 1.00 |
| What is relational algebra? | 0.33 | 1.00 |
| What is a SQL subquery and when would you use one? | 1.00 | 0.67 |
| What is MongoDB aggregation and how is it used? | 0.00 | 1.00 |
| What are Elasticsearch aggregations used for? | 0.67 | 0.67 |
| **Average** | **0.53** | **0.87** |

The sentence-level chunking strategy produced more precise retrievals overall. Its average Precision@3 was **0.87**, compared with **0.53** for fixed-size chunking. This suggests that grouping text by sentence boundaries helps preserve clearer semantic meaning, leading to more relevant retrieval results.

Fixed-size chunks sometimes retrieved unrelated surrounding content because the token window could mix multiple topics together. In contrast, sentence-level chunks were generally more focused and aligned with the questions, especially for relational algebra and MongoDB aggregation.

However, fixed-size chunking performed better for the SQL subquery question, where it achieved a higher Precision@3. Both strategies performed similarly for the Elasticsearch aggregation question. Overall, sentence-level chunking improved retrieval precision but did not completely eliminate retrieval noise.

### Building the RAG pipeline and a Streamlit chat app

In [ ]:
# Build Haystack RAG pipeline: BM25 retriever -> ChatPromptBuilder -> Qwen via HF Inference API.
from haystack import Pipeline
from haystack.components.builders import ChatPromptBuilder
from haystack.components.generators.chat import HuggingFaceAPIChatGenerator

document_store = make_document_store("transcripts_sentence")

template = """
{% message role=\"system\" %}
You are a helpful assistant.
{% endmessage %}

{% message role=\"user\" %}
Given the following information, answer the question.

Context:
{% for document in documents %}
    {{ document.content }} [Score: {{ document.score | round(3) }}]
{% endfor %}

Question: {{ query }}?
{% endmessage %}
"""

rag_pipeline = Pipeline()
rag_pipeline.add_component("retriever", ElasticsearchBM25Retriever(document_store=document_store, top_k=3))
rag_pipeline.add_component("prompt_builder", ChatPromptBuilder(template=template))
rag_pipeline.add_component("llm", HuggingFaceAPIChatGenerator(
    api_type="serverless_inference_api",
    api_params={"model": "meta-llama/Llama-3.2-1B-Instruct"},
))
rag_pipeline.connect("retriever.documents", "prompt_builder.documents")
rag_pipeline.connect("prompt_builder.prompt", "llm.messages")

# Quick test
test_q = "Give me everything that was covered for SQL"
result = rag_pipeline.run({"prompt_builder": {"query": test_q}, "retriever": {"query": test_q}})
try:
    print(result["llm"]["replies"][0].content)
except AttributeError:
    print(result["llm"]["replies"][0])

ChatMessage(_role=<ChatRole.ASSISTANT: 'assistant'>, _content=[TextContent(text='Based on the information provided, the following steps were covered for SQL three:\n\n1. Modifying the database to include only listings with prices that are multiples of five by changing the last review to "price" and copying the projection onto the database.\n\nThese are the main steps taken for SQL three as per the given information.')], _name=None, _meta={'model': 'meta-llama/Llama-3.2-1B-Instruct', 'finish_reason': 'stop', 'index': 0, 'usage': {'prompt_tokens': 284, 'completion_tokens': 65}})


In [ ]:
# Write app.py: Streamlit chatbot with BM25 retrieval and expandable retrieved-docs sidebar.
%%writefile app.py
import os
import streamlit as st
from haystack import Pipeline
from haystack_integrations.document_stores.elasticsearch import ElasticsearchDocumentStore
from haystack_integrations.components.retrievers.elasticsearch import ElasticsearchBM25Retriever
from haystack.components.builders import ChatPromptBuilder
from haystack.components.generators.chat import HuggingFaceAPIChatGenerator

ES_CLOUD_ID = os.environ.get("ES_CLOUD_ID")
ES_API_KEY  = os.environ.get("ES_API_KEY")
INDEX_NAME  = "transcripts_sentence"

@st.cache_resource
def build_rag_pipeline():
    document_store = ElasticsearchDocumentStore(
        hosts=None, cloud_id=ES_CLOUD_ID, api_key=ES_API_KEY, index=INDEX_NAME
    )
    template = """
{% message role="system" %}
You are a helpful assistant.
{% endmessage %}
{% message role="user" %}
Given the following information, answer the question.
Context:
{% for document in documents %}
    {{ document.content }} [Score: {{ document.score | round(3) }}]
{% endfor %}
Question: {{ query }}?
{% endmessage %}
"""
    pipeline = Pipeline()
    pipeline.add_component("retriever", ElasticsearchBM25Retriever(document_store=document_store, top_k=3))
    pipeline.add_component("prompt_builder", ChatPromptBuilder(template=template))
    pipeline.add_component("llm", HuggingFaceAPIChatGenerator(
        api_type="serverless_inference_api", api_params={"model": "meta-llama/Llama-3.2-1B-Instruct"} #"Qwen/Qwen2.5-7B-Instruct"
    ))
    pipeline.connect("retriever.documents", "prompt_builder.documents")
    pipeline.connect("prompt_builder.prompt", "llm.messages")
    return pipeline

rag_pipeline = build_rag_pipeline()

st.title("Course Chatbot")
st.caption("Interactive Q&A with Elasticsearch, Haystack, and HuggingFace")

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    st.chat_message(msg["role"]).write(msg["content"])

if prompt := st.chat_input("Ask a question about the course transcripts"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    st.chat_message("user").write(prompt)
    result = rag_pipeline.run({"prompt_builder": {"query": prompt}, "retriever": {"query": prompt}})
    try:
        response_text = result["llm"]["replies"][0].content
    except AttributeError:
        response_text = result["llm"]["replies"][0].text
    retrieved_docs = result.get("retriever", {}).get("documents", [])
    st.session_state.messages.append({"role": "assistant", "content": response_text})
    st.chat_message("assistant").write(response_text)
    if retrieved_docs:
        with st.expander("📄 Top 3 Retrieved Documents (BM25 Scores)"):
            for i, doc in enumerate(retrieved_docs):
                st.markdown(f"**Document {i+1}** — Source: `{doc.meta.get('source','N/A')}` | Chunk: `{doc.meta.get('chunk_id','N/A')}` | **BM25 Score: {doc.score:.3f}**")
                st.write(doc.content)
                st.divider()

Writing app.py


In [ ]:
# Export ES credentials as env vars, get LocalTunnel password, and launch Streamlit on port 8501.
import os
os.environ["ES_CLOUD_ID"] = ES_CLOUD_ID
os.environ["ES_API_KEY"]  = ES_API_KEY

# Get LocalTunnel password
!curl https://loca.lt/mytunnelpassword
print("\n")

# Launch Streamlit
!streamlit run app.py --server.enableCORS false --server.enableXsrfProtection false & npx localtunnel --port 8501

Password/endpoint: <redacted>


  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://<redacted>:8501
  External URL: http://<redacted>:8501

your url is: https://<redacted>.loca.lt


Here's what the deployed chatbot looked like, running through the localtunnel URL printed above:

![Course chatbot Streamlit interface, showing a question about SQL window functions and the model's answer](images/streamlit_chatbot.png)

### Fine-Tuning vs. RAG, head to head

In [ ]:
# Compare Fine-Tuning vs RAG

test_prompts = [
    "What are window functions in SQL?",
    "What is relational algebra?",
    "What is a SQL subquery and when would you use one?",
    "What is MongoDB aggregation and how is it used?",
    "What are Elasticsearch aggregations used for?"
]

for prompt in test_prompts:
    print("\n" + "="*70)
    print(f"Prompt: {prompt}")
    print("="*70)

    # ---------- Fine-tuned model ----------
    _messages = [
        {"role": "system",
         "content": "You are an instructor of CS 639 Data Management for Data Science course at UW-Madison."},
        {"role": "user", "content": prompt},
    ]

    _prompt = tokenizer.apply_chat_template(_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(_prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        out = trainer.model.generate(**inputs, max_new_tokens=200)

    response_ft = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    print("\nFine-tuned Model Response:\n")
    print(response_ft)

    # ---------- RAG ----------
    result = rag_pipeline.run({
        "prompt_builder": {"query": prompt},
        "retriever": {"query": prompt}
    })

    try:
        rag_response = result["llm"]["replies"][0].content
    except:
        rag_response = result["llm"]["replies"][0]

    print("\nRAG Response:\n")
    print(rag_response)


Prompt: What are window functions in SQL?

Fine-tuned Model Response:

Window functions in SQL are used to perform calculations over a set of rows that are related to a particular column or set of columns. They are defined using the `WHERE` clause and the `OVER` clause, which specifies the window over which the calculation is performed.

RAG Response:

ChatMessage(_role=<ChatRole.ASSISTANT: 'assistant'>, _content=[TextContent(text="Window functions in SQL are used to perform calculations or aggregations on a set of rows within a single row, resulting in a single row that contains the calculated values. Unlike aggregate functions, which also produce a single row, window functions produce a single row that contains the calculated values.\n\nWindow functions do not collapse the subset of rows into a single row, as seen with aggregate functions. Instead, they produce a new row that contains the calculated values for each row in the original set. This means that if there are ties or multip

**Which approach gave more accurate responses?**

Overall, the RAG approach produced more accurate and complete responses. The fine-tuned model often gave short or oversimplified answers, while RAG provided more detailed explanations and covered key concepts more thoroughly.

**Did the fine-tuned model hallucinate information?**

Yes, the fine-tuned model showed some hallucination and inaccuracies. For example, in the window functions question, it incorrectly mentioned the WHERE clause instead of focusing on OVER, which is essential. Some answers were also too vague or contained incorrect examples.

**Was RAG better at answering new or unseen questions?**

Yes, RAG performed better on new or unseen questions. Since it retrieves relevant context from transcripts, it can generate more grounded and context-aware answers. In contrast, the fine-tuned model relies only on its learned patterns, which makes it less reliable for unfamiliar queries.